# RE_Intgrtn1 - Example 4 (Perfect Dispatch)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

> Note: there is no `SUC_IC_e4.mod` in the original AMPL distribution. This notebook is a Pyomo implementation of the **Perfect Dispatch** example added in the lecture-15 slides, demonstrating the cost gap between a stochastic UC made under uncertainty and a perfect-foresight dispatch.

Same generator/load data as Example 3 (`RE_Intgrtn1_e3_SUC_data.txt`):
- load = 100 MW
- forecasted solar: 60 (25 %), 40 (50 %), 20 (25 %)

But now we assume the **actual realized solar power is 20 MW** (the worst-case scenario). Compare two strategies:

| Strategy | What it does | Slide TC |
|---|---|---|
| **S-UC then realize** | Solve S-UC ahead of time (only forecast known), take its commitment (`u1=0, u2=1`, from Example 3), apply it when actual solar = 20 MW. | **$1700** |
| **Perfect Dispatch** | Solve D-UC with full knowledge of actual solar = 20 MW. | **$1600** |

The $100 gap is the **cost of uncertainty**.


In [1]:
# ---- Load the same data file as Example 3 ----
import sys, pathlib
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('RE_Intgrtn1_e3_SUC_data.txt')
GEN_data    = _d['GEN']
gen_min     = _d['gen_min']
gen_max     = _d['gen_max']
gen_OpCost  = _d['gen_OpCost']
gen_SuCost  = _d['gen_SuCost']
TotalLoad   = _d['Time_TotalPd'][1]       # = 100 MW

ACTUAL_SOLAR  = 20                        # MW (the worst-case realization)
actual_netload = TotalLoad - ACTUAL_SOLAR  # = 80 MW

print(f'Total load = {TotalLoad} MW,  actual solar = {ACTUAL_SOLAR} MW')
print(f'Actual netload = {actual_netload} MW')


Total load = 100 MW,  actual solar = 20 MW
Actual netload = 80 MW


In [2]:
# ---- Strategy 1: S-UC commitment applied to the realized scenario ----
# From RE_Intgrtn1_e3_SUC.ipynb: S-UC chose u1 = 0, u2 = 1
from pyomo.environ import (
    ConcreteModel, Var, Constraint, Objective, SolverFactory,
    NonNegativeReals, minimize, value
)

u_SUC = {1: 0, 2: 1}        # commitment from Example 3 (S-UC solution)

m1 = ConcreteModel()
m1.Pg = Var(GEN_data, domain=NonNegativeReals,
            bounds=lambda mm, g: (gen_min[g]*u_SUC[g], gen_max[g]*u_SUC[g]))
m1.balance = Constraint(expr=sum(m1.Pg[g] for g in GEN_data) == actual_netload)
m1.obj = Objective(
    expr=sum(gen_OpCost[g]*m1.Pg[g] + gen_SuCost[g]*u_SUC[g] for g in GEN_data),
    sense=minimize,
)
SolverFactory('gurobi').solve(m1)

print('=== Strategy 1: S-UC commitment + realized actual solar ===')
for g in GEN_data:
    print(f'  u{g} = {u_SUC[g]},  G{g} = {value(m1.Pg[g]):g} MW')
TC_SUC = value(m1.obj)
print(f'  TC = ${TC_SUC:g}')


=== Strategy 1: S-UC commitment + realized actual solar ===
  u1 = 0,  G1 = 0 MW
  u2 = 1,  G2 = 80 MW
  TC = $1700


In [3]:
# ---- Strategy 2: Perfect Dispatch (D-UC with full knowledge of actual solar) ----
from pyomo.environ import Binary

m2 = ConcreteModel()
m2.u  = Var(GEN_data, domain=Binary)
m2.Pg = Var(GEN_data, domain=NonNegativeReals)
m2.balance = Constraint(expr=sum(m2.Pg[g] for g in GEN_data) == actual_netload)
m2.gmin = Constraint(GEN_data, rule=lambda mm, g: mm.Pg[g] >= gen_min[g]*mm.u[g])
m2.gmax = Constraint(GEN_data, rule=lambda mm, g: mm.Pg[g] <= gen_max[g]*mm.u[g])
m2.obj = Objective(
    expr=sum(gen_OpCost[g]*m2.Pg[g] + gen_SuCost[g]*m2.u[g] for g in GEN_data),
    sense=minimize,
)
SolverFactory('gurobi').solve(m2)

print('=== Strategy 2: Perfect Dispatch (D-UC with actual solar known) ===')
for g in GEN_data:
    print(f'  u{g} = {int(round(value(m2.u[g])))},  G{g} = {value(m2.Pg[g]):g} MW')
TC_PD = value(m2.obj)
print(f'  TC = ${TC_PD:g}')


=== Strategy 2: Perfect Dispatch (D-UC with actual solar known) ===
  u1 = 1,  G1 = 80 MW
  u2 = 0,  G2 = 0 MW
  TC = $1600


In [4]:
# ---- Comparison ----
print('=== Comparison (lecture 15, slide page 36) ===')
print(f'  S-UC realized cost   = ${TC_SUC:g}')
print(f'  Perfect Dispatch     = ${TC_PD:g}')
print(f'  Cost of uncertainty  = ${TC_SUC - TC_PD:g}')


=== Comparison (lecture 15, slide page 36) ===
  S-UC realized cost   = $1700
  Perfect Dispatch     = $1600
  Cost of uncertainty  = $100
